# Staleness and recompute — sensing a derived table's drift, and repairing it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/20-recompute/recompute.ipynb)

Built from [`cookbook/book/chapters/20-recompute/recompute.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/20-recompute/recompute.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `generate_embeddings` → `build_neighbor_graph` → `propagate_embeddings`
under `cache="use"` · `describe_table` · `staleness` · `derives_from` ·
`refresh_embeddings` · `recompute(table, cascade=…)` · `verify_materialization` ·
`list_jobs` / `job` · **Theory:** a materialized artifact is the output of a
recorded *definition* over a recorded *input state* (Kleppmann 2017); it is
fresh while both still hold, and a change to either is detectable ·
**Rail:** measurement (every outcome observed live on the papers).

A derived table is only as current as what it was built from. When the source
under an embedding table changes, the neighbour graph built on those embeddings,
and the propagation built on that graph, are silently out of date — unless the
engine can say so. It can, because every result table records how it was
produced: its **definition** (a hash of the producing parameters, the model and
the environment) and an **anchor** for each input (a result table's content
digest, or a source's read instant). This chapter builds a three-step chain over
the papers, then walks through what the engine senses and how it repairs:

1. **Reuse** — a producer asked for an exact derivation it already has returns it.
2. **A changed definition** — asked about a different definition, a table is stale.
3. **A changed input** — the source changes, the embeddings refresh, and
   everything derived from them senses the drift and is recomputed, once, on
   request.

In [ ]:
import tempfile
from pathlib import Path

import jammi
import pyarrow as pa
import pyarrow.parquet as pq
from jammi_cookbook import datasets, encoders, scale

SCALE = scale.current()
MODEL = encoders.text(SCALE)
work = Path(tempfile.mkdtemp())
db = jammi.connect(f"file://{tempfile.mkdtemp()}")

# The papers, as a source this chapter owns — its file is what changes below.
arxiv = datasets.arxiv(db, SCALE)
papers = db.sql(f"SELECT paper_id, title, abstract FROM {arxiv.papers}.public.{arxiv.papers}")
pq.write_table(papers, work / "papers.parquet")
db.add_source("papers", url=str(work / "papers.parquet"), format="parquet")

embeddings = db.generate_embeddings(source="papers", model=MODEL,
                                    columns=["title", "abstract"], key="paper_id")
graph = db.build_neighbor_graph("papers", k=10, embedding_table=embeddings, cache="use")
propagated = db.propagate_embeddings("papers", embedding_table=embeddings,
                                     edge_graph_table=graph, hops=2, alpha=0.1,
                                     output="final", cache="use")
print(f"embeddings → {graph[:40]}… → {propagated[:40]}…")

## Reuse — an exact derivation is returned, not recomputed

With `cache="use"`, a producer first looks for a table it already made from the
same definition over the same inputs, and returns it. Table names make this
visible: a reuse returns the *same* name, a computation a new one.

In [ ]:
outcomes = {
    "the same neighbour graph": db.build_neighbor_graph(
        "papers", k=10, embedding_table=embeddings, cache="use") == graph,
    "a graph with k = 5": db.build_neighbor_graph(
        "papers", k=5, embedding_table=embeddings, cache="use") == graph,
    "the same propagation": db.propagate_embeddings(
        "papers", embedding_table=embeddings, edge_graph_table=graph, hops=2, alpha=0.1,
        output="final", cache="use") == propagated,
    "the same embeddings": db.generate_embeddings(
        source="papers", model=MODEL, columns=["title", "abstract"], key="paper_id",
        cache="use") == embeddings,
}
for what, reused in outcomes.items():
    print(f"  {what:<26} reused: {reused}")

In [ ]:
assert outcomes == {"the same neighbour graph": True, "a graph with k = 5": False,
                    "the same propagation": True, "the same embeddings": False}

Each request names the embedding table it reads: the propagation above published a
second embedding table on the same source, and a request that named none would read
the newest. The graph and the propagation read only result tables, whose content
digests pin their inputs exactly, so an identical request is safely reused and a changed
parameter (`k`) computes anew. The embeddings are different: they read a *file*
source, which has no version the engine can pin — a file can change in place —
so their reuse can never be proven safe, and they are always recomputed.

## A changed definition

`staleness(table, definition)` compares a table's recorded definition, and each
recorded input anchor, with the present. Asked about its own recorded definition,
the propagation is fresh; asked about another — the definition a changed model,
parameter or environment would produce — it is stale, and says why.

In [ ]:
recorded = db.describe_table(propagated)["definition_hash"]
fresh = db.staleness(propagated, recorded)
changed = db.staleness(propagated, "0" * len(recorded))
print(f"against its recorded definition: {fresh['staleness']}")
print(f"against another definition:      {changed['staleness']}  "
      f"— {[r['reason'] for r in changed['reasons']]}")

In [ ]:
assert fresh["staleness"] == "fresh"
assert changed["staleness"] == "stale"
assert [r["reason"] for r in changed["reasons"]] == ["definition_changed"]

## A changed input — sense the drift, then recompute

Now the source changes: one paper's abstract is corrected. `refresh_embeddings`
re-embeds only the rows whose content changed and publishes a new version of the
embedding table, so its content digest moves. The graph was built over the old
digest, and the engine senses it.

In [ ]:
changed_papers = papers.to_pylist()
changed_papers[0]["abstract"] = changed_papers[0]["abstract"] + " (corrected)"
pq.write_table(pa.Table.from_pylist(changed_papers, schema=papers.schema),
               work / "papers.parquet")
report = db.refresh_embeddings(embeddings)
print(f"refresh: {report['outcome']} — {report['inferred_rows']} row(s) re-embedded")

graph_definition = db.describe_table(graph)["definition_hash"]
drift = db.staleness(graph, graph_definition)
print(f"the graph, against its own definition: {drift['staleness']}  "
      f"— {[r['reason'] for r in drift['reasons']]}")
print(f"derived from the embeddings: "
      f"{sorted(e['derived'][:40] for e in db.derives_from(embeddings))}")

In [ ]:
assert report["outcome"] == "published" and report["inferred_rows"] == 1
assert drift["staleness"] == "stale"
assert [r["reason"] for r in drift["reasons"]] == ["input_advanced"]

`derives_from` names what is built on a table, one hop at a time. The repair is
one explicit call: `recompute(graph, cascade="downstream")` re-runs the graph's
recorded definition over the embeddings' current version, then sweeps what was
built on the graph — the propagation — once.

In [ ]:
repair = db.recompute(graph, cascade="downstream")
for step in repair["recomputed"]:
    print(f"  {step['original'][:40]}… → {step['recomputed'][:40]}…  "
          f"({step['outcome']['outcome']})")
new_graph = repair["recomputed"][0]["recomputed"]
print(f"the recomputed graph: "
      f"{db.staleness(new_graph, db.describe_table(new_graph)['definition_hash'])['staleness']}")

In [ ]:
assert len(repair["recomputed"]) == 2
assert all(step["outcome"]["outcome"] == "computed" for step in repair["recomputed"])
assert db.staleness(new_graph, db.describe_table(new_graph)["definition_hash"])["staleness"] == "fresh"

A recompute over inputs that have *not* moved produces the same artifact, and
`verify_materialization` attests it against the original definition:

In [ ]:
again = db.recompute(new_graph, cascade="report_only")["recomputed"][0]["recomputed"]
print(f"recompute over unmoved inputs: "
      f"{db.verify_materialization(again, expected_definition=db.describe_table(new_graph)['definition_hash'])['verdict']}")

In [ ]:
assert db.verify_materialization(
    again, expected_definition=db.describe_table(new_graph)["definition_hash"])["verdict"] == "match"

## The job queue underneath

Every producer above ran as a job: the same durable queue a fine-tune goes
through, recorded and claimed in the caller's own process for a synchronous
call. `list_jobs` reads them back, and `job(id)` reattaches to one.

In [ ]:
jobs = db.list_jobs()
kinds = sorted({j["kind"] for j in jobs})
print(f"{len(jobs)} jobs of kinds {kinds}")
latest = db.job(jobs[0]["job_id"])
print(f"job {jobs[0]['job_id'][:8]}…: {latest.status()}")
db.close()

In [ ]:
assert {"embedding", "neighbor_graph", "propagate"} <= set(kinds)

## The boundary — mechanism, not the loop

The engine ships the bounded **mechanism**: one cache probe per producer call, one
staleness read per question, one recompute on one explicit request, one bounded
downstream sweep on `cascade="downstream"`. It ships no scheduler and no monitor
that triggers a recompute on its own: wiring the sensor (`staleness`) to the
actuator (`recompute`) on a schedule is the consumer's composition. One bounded
sweep on one explicit request is engine; re-running it on a schedule is a
platform built on it.

## References

- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.